In [1]:
import os
label_dir = r'C:\Users\rishi\Downloads\dataset_oil\oil-spill\train\labels'
print("Folder exists:", os.path.exists(label_dir))
files = os.listdir(label_dir)
print("Number of files:", len(files))
print("First 5 files:", files[:5])

Folder exists: True
Number of files: 1002
First 5 files: ['img_0001.png', 'img_0002.png', 'img_0003.png', 'img_0004.png', 'img_0005.png']


In [ ]:
from PIL import Image
import numpy as np
import os

label_dir = r'C:\Users\rishi\Downloads\dataset_oil\oil-spill\train\labels'
all_colors = set()
count = 0

for file in os.listdir(label_dir):
    if not file.endswith(('.png', '.jpg', '.jpeg')):
        continue
    full_path = os.path.join(label_dir, file)
    img = np.array(Image.open(full_path).convert('RGB')).astype(int)
    pixels = img.reshape(-1, 3)
    unique = set(map(tuple, pixels))
    all_colors.update(unique)
    count += 1

print(f"Total files processed: {count}")

print(f"Total unique colors: {len(all_colors)}")
for color in sorted(all_colors):
    print(f"  RGB{color}")

In [4]:
import cv2
import numpy as np
import os

label_dir  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\labels'
output_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\binary_mask'
os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(label_dir):
    if not file.endswith('.png'):
        continue

    img = cv2.imread(os.path.join(label_dir, file))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lower_oil = np.array([0,   200, 200])
    upper_oil = np.array([50,  255, 255])
    oil_mask  = cv2.inRange(img, lower_oil, upper_oil)

    binary_label = (oil_mask / 255).astype(np.uint8)
    visual_mask = binary_label * 255

    cv2.imwrite(os.path.join(output_dir, file), visual_mask)

count = len(os.listdir(output_dir))
print(f" {count} masks saved")


 1002 masks saved


In [5]:

mask_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\binary_mask'

total_files = 0
has_oil = 0
no_oil = 0
corrupt = 0

for file in os.listdir(mask_dir):
    if not file.endswith('.png'):
        continue
    
    total_files += 1
    mask = cv2.imread(os.path.join(mask_dir, file), 0)
    
    if mask is None:
        corrupt += 1
        continue
    
    if np.any(mask == 255):
        has_oil += 1
    else:
        no_oil += 1

print(f"Total masks     : {total_files}")
print(f"Has oil (white) : {has_oil}")
print(f"No oil          : {no_oil}")
print(f"Corrupt files   : {corrupt}")
print(f"Oil %           : {has_oil/total_files*100:.1f}%")



Total masks     : 1002
Has oil (white) : 791
No oil          : 211
Corrupt files   : 0
Oil %           : 78.9%


Lee Filter (Speckle Noise Reduction)

In [ ]:
from scipy.ndimage import uniform_filter

input_dir  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\images'
output_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\filtered_images'
os.makedirs(output_dir, exist_ok=True)

def lee_filter(img, size=7):
    img = img.astype(np.float64)
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img ** 2, size)
    img_variance = img_sqr_mean - img_mean ** 2
    overall_variance = np.var(img)

    weights = img_variance / (img_variance + overall_variance + 1e-10)
    filtered = img_mean + weights * (img - img_mean)

    return np.clip(filtered, 0, 255).astype(np.uint8)

files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for idx, file in enumerate(files):
    img = cv2.imread(os.path.join(input_dir, file), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        continue
        
    filtered = lee_filter(img)
    cv2.imwrite(os.path.join(output_dir, file), filtered)

    if (idx + 1) % 100 == 0:
        print(f"{idx+1}/{len(files)} done")

print("finished")

Lee Filter Visualization

In [ ]:
import matplotlib.pyplot as plt

original = cv2.imread(r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\images\img_0003.jpg', 0)
filtered = cv2.imread(r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\filtered_images\img_0003.jpg', 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original (noisy)')

axes[1].imshow(filtered, cmap='gray')
axes[1].set_title('After Lee Filter (smooth)')

plt.show()

Image Normalization

In [ ]:
input_dir  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\filtered_images'
output_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\normalized_images'
os.makedirs(output_dir, exist_ok=True)

files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for idx, file in enumerate(files):

    img = cv2.imread(os.path.join(input_dir, file), cv2.IMREAD_GRAYSCALE)

    if img is None:
        continue

    normalized = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)

    cv2.imwrite(os.path.join(output_dir, file), normalized)

    if (idx + 1) % 100 == 0:
        print(f"{idx+1}/{len(files)} done")

print("finished")

Image & Mask Resizing

In [ ]:
images_dir = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\normalized_images'
masks_dir  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\binary_mask'

output_images = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\resized_images'
output_masks  = r'C:\Users\DELL\OneDrive\Desktop\Oil_Spill\oil-spill\train\resized_masks'

os.makedirs(output_images, exist_ok=True)
os.makedirs(output_masks, exist_ok=True)

files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]

for idx, file in enumerate(files):

    base = os.path.splitext(file)[0]

    img  = cv2.imread(os.path.join(images_dir, file), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(os.path.join(masks_dir, base + '.png'), cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        continue

    img_resized  = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
    mask_resized = cv2.resize(mask, (512, 512), interpolation=cv2.INTER_NEAREST)

    cv2.imwrite(os.path.join(output_images, base + '.png'), img_resized)
    cv2.imwrite(os.path.join(output_masks, base + '.png'), mask_resized)

    if (idx + 1) % 100 == 0:
        print(f"{idx+1}/{len(files)} done")
print("finished")